<a href="https://colab.research.google.com/github/jennyk23/Magpy/blob/main/atividade6_media_status_aluno_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade 6 — Média e status do aluno
Execute a célula abaixo. Ela instala o MariaDB, cria o banco, cria as functions e mostra os resultados.

In [1]:
import os
import shutil
import subprocess
import time
from pathlib import Path

if shutil.which("mariadb") is None:
    print("Instalando MariaDB...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "mariadb-server", "mariadb-client"],
        check=True,
        stdout=subprocess.DEVNULL
    )

os.makedirs("/run/mysqld", exist_ok=True)
subprocess.run(["chown", "mysql:mysql", "/run/mysqld"], check=False)
subprocess.run(
    ["service", "mariadb", "start"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(20):
    teste = subprocess.run(
        ["mariadb-admin", "-u", "root", "ping", "--silent"],
        capture_output=True,
        text=True
    )
    if teste.returncode == 0:
        break
    time.sleep(1)
else:
    raise RuntimeError("Não foi possível iniciar o MariaDB.")

print("MariaDB iniciado com sucesso.")

sql = "-- ============================================================\n-- ATIVIDADE 6 – MÉDIA E STATUS DO ALUNO COM FUNCTIONS\n-- MYSQL / MARIADB\n-- ============================================================\n\nDROP DATABASE IF EXISTS atividade6_alunos;\n\nCREATE DATABASE atividade6_alunos\n    CHARACTER SET utf8mb4\n    COLLATE utf8mb4_unicode_ci;\n\nUSE atividade6_alunos;\n\nCREATE TABLE aluno (\n    id_aluno INT AUTO_INCREMENT PRIMARY KEY,\n    nome VARCHAR(150) NOT NULL,\n    CONSTRAINT chk_aluno_nome\n        CHECK (CHAR_LENGTH(TRIM(nome)) > 0)\n) ENGINE = InnoDB;\n\nCREATE TABLE nota (\n    id_nota INT AUTO_INCREMENT PRIMARY KEY,\n    id_aluno INT NOT NULL,\n    valor DECIMAL(4,2) NOT NULL,\n\n    CONSTRAINT fk_nota_aluno\n        FOREIGN KEY (id_aluno)\n        REFERENCES aluno(id_aluno)\n        ON UPDATE CASCADE\n        ON DELETE CASCADE,\n\n    CONSTRAINT chk_nota_valor\n        CHECK (valor BETWEEN 0 AND 10)\n) ENGINE = InnoDB;\n\nINSERT INTO aluno (nome) VALUES\n('Ana Beatriz Souza'),\n('Bruno Henrique Lima'),\n('Carla Mendes Rocha');\n\nINSERT INTO nota (id_aluno, valor) VALUES\n(1, 8.00),\n(1, 9.00),\n(2, 5.50),\n(2, 7.00),\n(3, 6.00),\n(3, 8.00);\n\nDROP FUNCTION IF EXISTS media_aluno;\nDROP FUNCTION IF EXISTS status_aluno;\n\nDELIMITER $$\n\nCREATE FUNCTION media_aluno(p_id INT)\nRETURNS DECIMAL(4,2)\nREADS SQL DATA\nBEGIN\n    DECLARE v_aluno_existe INT DEFAULT 0;\n    DECLARE v_media DECIMAL(4,2) DEFAULT 0.00;\n\n    SELECT COUNT(*)\n    INTO v_aluno_existe\n    FROM aluno\n    WHERE id_aluno = p_id;\n\n    IF v_aluno_existe = 0 THEN\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT = 'Aluno não encontrado.';\n    END IF;\n\n    SELECT COALESCE(ROUND(AVG(valor), 2), 0.00)\n    INTO v_media\n    FROM nota\n    WHERE id_aluno = p_id;\n\n    RETURN v_media;\nEND$$\n\nCREATE FUNCTION status_aluno(p_id INT)\nRETURNS VARCHAR(20)\nREADS SQL DATA\nBEGIN\n    DECLARE v_media DECIMAL(4,2);\n\n    SET v_media = media_aluno(p_id);\n\n    IF v_media >= 7 THEN\n        RETURN 'Aprovado';\n    ELSE\n        RETURN 'Reprovado';\n    END IF;\nEND$$\n\nDELIMITER ;\n\nSELECT\n    a.id_aluno,\n    a.nome,\n    media_aluno(a.id_aluno) AS media,\n    status_aluno(a.id_aluno) AS status\nFROM aluno AS a\nORDER BY a.id_aluno;\n\nSHOW FUNCTION STATUS\nWHERE Db = 'atividade6_alunos';\n"
arquivo_sql = Path("/content/atividade6_media_status_aluno.sql")
arquivo_sql.write_text(sql, encoding="utf-8")
print("Arquivo SQL criado em:", arquivo_sql)

with arquivo_sql.open("r", encoding="utf-8") as arquivo:
    execucao = subprocess.run(
        ["mariadb", "-u", "root", "--default-character-set=utf8mb4"],
        stdin=arquivo,
        capture_output=True,
        text=True
    )

if execucao.returncode != 0:
    print(execucao.stderr)
    raise RuntimeError("Erro ao executar o script SQL.")

consultas = "USE atividade6_alunos;\n\nSELECT\n    'RESULTADO DAS FUNÇÕES' AS etapa;\n\nSELECT\n    a.id_aluno,\n    a.nome,\n    media_aluno(a.id_aluno) AS media,\n    status_aluno(a.id_aluno) AS status\nFROM aluno AS a\nORDER BY a.id_aluno;\n\nSELECT\n    'NOTAS CADASTRADAS' AS etapa;\n\nSELECT\n    a.nome AS aluno,\n    n.id_nota,\n    n.valor\nFROM aluno AS a\nINNER JOIN nota AS n\n    ON n.id_aluno = a.id_aluno\nORDER BY a.id_aluno, n.id_nota;\n\nSELECT\n    'FUNÇÕES CRIADAS' AS etapa;\n\nSHOW FUNCTION STATUS\nWHERE Db = 'atividade6_alunos';\n"
resultado = subprocess.run(
    [
        "mariadb",
        "-u",
        "root",
        "--table",
        "--default-character-set=utf8mb4",
        "-e",
        consultas
    ],
    capture_output=True,
    text=True
)

print(resultado.stdout)

if resultado.returncode != 0:
    print(resultado.stderr)
    raise RuntimeError("Erro ao consultar os resultados.")

print("Atividade 6 executada com sucesso.")


Instalando MariaDB...
MariaDB iniciado com sucesso.
Arquivo SQL criado em: /content/atividade6_media_status_aluno.sql
+-------------------------+
| etapa                   |
+-------------------------+
| RESULTADO DAS FUNÇÕES   |
+-------------------------+
+----------+---------------------+-------+-----------+
| id_aluno | nome                | media | status    |
+----------+---------------------+-------+-----------+
|        1 | Ana Beatriz Souza   |  8.50 | Aprovado  |
|        2 | Bruno Henrique Lima |  6.25 | Reprovado |
|        3 | Carla Mendes Rocha  |  7.00 | Aprovado  |
+----------+---------------------+-------+-----------+
+-------------------+
| etapa             |
+-------------------+
| NOTAS CADASTRADAS |
+-------------------+
+---------------------+---------+-------+
| aluno               | id_nota | valor |
+---------------------+---------+-------+
| Ana Beatriz Souza   |       1 |  8.00 |
| Ana Beatriz Souza   |       2 |  9.00 |
| Bruno Henrique Lima |       3 |  5.